# Fermionic h=0 architecture ladder — presentation figure

Three tiers at $h_x=h_z=0$, $L\in\{2,3,4\}$, multi-seed:
**CNN** (GeoCNN, symmetry-unaware, complex) vs **Approx. Symm.** (`ToricCNN_gridinv`, no
sign head, cold start) vs **Sign-Head + Approx. Symm.** (frozen analytic GF(2) head +
flux penalty + `chains_up`). Exact anchor $E_0=-4L^3$ (per spin $-4/3$).

Data: `results/fermionic_ladder/*.curve.json` (rsynced from
`$PSCRATCH/tc_nqs/fermionic_ladder/`, produced by `nersc/launch_fermionic_ladder.sh`).

In [ ]:
# %% 1. CONFIG ------------------------------------------------------------
import json, glob, os, re
import numpy as np
import matplotlib.pyplot as plt

BASE   = "../results/fermionic_ladder"
FIGS   = "figs"
LS     = [2, 3, 4]
E0     = {L: -4 * L**3 for L in LS}          # exact h=0 PBC anchor
NSPIN  = {L: 3 * L**3 for L in LS}
TRAP_L2 = -22.521                            # positive-sector optimum (BLOG 2026-08-07)

# tier -> (glob stem, label, plasma color); tweak labels for the talk here
TIERS = {
    "cnn":      ("ladder_cnn_L{L}_*",      "CNN (no symmetry)",          plt.cm.plasma(0.08)),
    "asymm":    ("ladder_asymm_L{L}_*",    "Approx. Symm.",              plt.cm.plasma(0.55)),
    "signhead": ("ladder_signhead_L{L}_*", "Sign-Head + Approx. Symm.",  plt.cm.plasma(0.85)),
}
VARIANT_LS = {"ch444": "-", "ch88": "--", "inv88": "-", "inv222": "--", "": "-"}
ERR_FLOOR  = 1e-9                            # log-panel clip (sampling-resolution floor)

plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
# %% 2. load curves --------------------------------------------------------
# name convention: ladder_{tier}_L{L}_{variant}_s{seed}[ _smoke ]  (variant absent for signhead)
PAT = re.compile(r"ladder_(?P<tier>cnn|asymm|signhead)_L(?P<L>\d)(?:_(?P<var>[a-z0-9]+))?_s(?P<seed>\d+)")
runs = {}                                     # (tier, L, var, seed) -> dict(step, E)
for path in sorted(glob.glob(os.path.join(BASE, "ladder_*.curve.json"))):
    name = os.path.basename(path).replace(".curve.json", "")
    if name.endswith("_smoke"):
        continue
    m = PAT.match(name)
    if not m:
        print("skip (unparsed):", name); continue
    d = json.load(open(path))["curve"]
    step = np.asarray(d["step"], float); E = np.asarray(d["energy"], float)
    o = np.argsort(step); step, E = step[o], E[o]          # resumes: sort + dedupe
    keep = np.concatenate([[True], np.diff(step) > 0]); step, E = step[keep], E[keep]
    runs[(m["tier"], int(m["L"]), m["var"] or "", int(m["seed"]))] = {"step": step, "E": E}
print(f"{len(runs)} runs loaded")
for L in LS:
    counts = {t: sum(1 for k in runs if k[0] == t and k[1] == L) for t in TIERS}
    print(f"  L={L}: {counts}")

In [ ]:
# %% 3. the figure: E/N (top) + log10 relative error (bottom) --------------
fig, axes = plt.subplots(2, len(LS), figsize=(13.5, 7.2), sharex="col",
                         gridspec_kw={"height_ratios": [1.25, 1.0], "hspace": 0.08})
for j, L in enumerate(LS):
    at, ab = axes[0, j], axes[1, j]
    for tier, (_, label, col) in TIERS.items():
        first = True
        for (t, Lk, var, seed), r in sorted(runs.items()):
            if t != tier or Lk != L: continue
            eN   = r["E"] / NSPIN[L]
            rel  = np.clip(np.abs(r["E"] - E0[L]) / abs(E0[L]), ERR_FLOOR, None)
            lab  = label if first else None; first = False
            ls   = VARIANT_LS.get(var, "-")
            at.plot(r["step"], eN, color=col, lw=1.4, ls=ls, alpha=0.85, label=lab)
            ab.plot(r["step"], rel, color=col, lw=1.4, ls=ls, alpha=0.85)
    at.axhline(E0[L] / NSPIN[L], color="k", ls="--", lw=1.2)
    if L == 2:
        at.axhline(TRAP_L2 / NSPIN[2], color="grey", ls=":", lw=1.2)
        at.text(0.98, TRAP_L2 / NSPIN[2] + 0.012, "positive-sector optimum",
                color="grey", fontsize=8, ha="right", transform=at.get_yaxis_transform())
    at.set_title(f"$L={L}$  ($N={NSPIN[L]}$ spins, $E_0=-4L^3={E0[L]}$)")
    ab.set_yscale("log"); ab.set_xlabel("SR step")
    ab.axhline(ERR_FLOOR, color="none")
    if j == 0:
        at.set_ylabel(r"$\langle H \rangle / N$")
        ab.set_ylabel(r"$|E - E_0| / |E_0|$")
        at.legend(frameon=False, fontsize=9, loc="upper right")
at2 = axes[0, 0]
at2.text(0.02, 0.03, "dashed: exact $E_0/N=-4/3$", transform=at2.transAxes,
         fontsize=8, color="k")
fig.suptitle("fermionic 3D toric code, $h=0$ — what the sign head buys, across architectures",
             y=0.98)
# plt.savefig(os.path.join(FIGS, "fermionic_arch_ladder.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# %% 4. plateau table -------------------------------------------------------
rows = []
for (t, L, var, seed), r in sorted(runs.items()):
    tail = r["E"][-20:]
    rows.append((t, L, var or "-", seed, np.mean(tail),
                 abs(np.mean(tail) - E0[L]) / abs(E0[L])))
print(f"{'tier':10} {'L':>2} {'variant':8} {'seed':>4} {'E (last-20 mean)':>18} {'rel. err':>10}")
for t, L, var, seed, e, d in rows:
    print(f"{t:10} {L:>2} {var:8} {seed:>4} {e:>18.6f} {d:>10.2e}")